## 1. Ingestion

In [2]:
import os, json, time, requests
import pandas as pd
from dotenv import load_dotenv

# https://pypi.org/project/python-dotenv/
load_dotenv(dotenv_path="../.env")
URL = f"https://eth-mainnet.g.alchemy.com/v2/{os.getenv('ALCHEMY_KEY')}"

def rpc(method, params, retries=3):
    for attempt in range(retries):
        try:
            # jsonrpc.org/specification
            r = requests.post(URL, json={"jsonrpc": "2.0", "id": 1, "method": method, "params": params}, timeout=30)
            result = r.json()
            if "error" in result:
                raise Exception(result["error"])
            return result["result"]
        except Exception as e:
            if attempt == retries - 1:
                raise
            print(f"  retry {attempt+1} after error: {e}")
            time.sleep(2 ** attempt)

latest = int(rpc("eth_blockNumber", []), 16)
print("Latest block:", latest)

Latest block: 25045360


In [3]:
# Define fixed range — hardcode for reproducibility
END_BLOCK = latest - 10           # small safety buffer from the head
START_BLOCK = END_BLOCK - 999     # 1000 blocks total
print(f"Pulling blocks {START_BLOCK} to {END_BLOCK} ({END_BLOCK - START_BLOCK + 1} blocks)")

Pulling blocks 25044351 to 25045350 (1000 blocks)


In [ ]:
# Pull all blocks with full transactions
tx_rows = []
t0 = time.time()

for n in range(START_BLOCK, END_BLOCK + 1):
    cache_path = f"../data/raw/block_{n}.json"
    if os.path.exists(cache_path):
        with open(cache_path) as f:
            b = json.load(f)
    else:
        b = rpc("eth_getBlockByNumber", [hex(n), True])
        with open(cache_path, "w") as f:
            json.dump(b, f)
        time.sleep(0.05)

    ts = int(b["timestamp"], 16)
    for tx in b["transactions"]:
        tx_rows.append({
            "block": n,
            "ts": ts,
            "tx_hash": tx["hash"],
            "from": tx["from"],
            "to": tx["to"],
            "value_wei": int(tx["value"], 16),
            "gas": int(tx["gas"], 16),
            "input_len": len(tx["input"]),
            "is_contract_call": tx["input"] != "0x",
        })

    if (n - START_BLOCK) % 50 == 0:
        elapsed = time.time() - t0
        print(f"  block {n} ({n - START_BLOCK + 1}/{END_BLOCK - START_BLOCK + 1}) — {elapsed:.0f}s elapsed")

print(f"\nDone. {len(tx_rows)} transactions in {time.time()-t0:.0f}s")

  block 25044351 (1/1000) — 1s elapsed
  block 25044401 (51/1000) — 49s elapsed
  block 25044451 (101/1000) — 97s elapsed
  block 25044501 (151/1000) — 145s elapsed
  block 25044551 (201/1000) — 193s elapsed
  block 25044601 (251/1000) — 241s elapsed
  block 25044651 (301/1000) — 290s elapsed
  block 25044701 (351/1000) — 339s elapsed
  block 25044751 (401/1000) — 388s elapsed
  block 25044801 (451/1000) — 437s elapsed
  block 25044851 (501/1000) — 487s elapsed
  block 25044901 (551/1000) — 534s elapsed
  block 25044951 (601/1000) — 582s elapsed
  block 25045001 (651/1000) — 631s elapsed
  block 25045051 (701/1000) — 680s elapsed
  block 25045101 (751/1000) — 730s elapsed
  block 25045151 (801/1000) — 781s elapsed
  block 25045201 (851/1000) — 830s elapsed
  block 25045251 (901/1000) — 877s elapsed
  block 25045301 (951/1000) — 925s elapsed

Done. 265653 transactions in 972s


In [ ]:
# Build transactions DataFrame, save
tx_df = pd.DataFrame(tx_rows)
tx_df["value_eth"] = tx_df["value_wei"] / 1e18
tx_df["dt"] = pd.to_datetime(tx_df["ts"], unit="s")

tx_df.to_csv("../data/processed/transactions.csv", index=False)
print(f"Saved {len(tx_df):,} txs")
print(f"Unique senders: {tx_df['from'].nunique():,}")
print(f"Unique recipients: {tx_df['to'].nunique():,}")
print(f"Contract call ratio: {tx_df['is_contract_call'].mean():.2%}")
tx_df.head()

Saved 265,653 txs
Unique senders: 89,622
Unique recipients: 52,579
Contract call ratio: 71.06%


,block,ts,tx_hash,from,to,value_wei,gas,input_len,is_contract_call,value_eth,dt
0,25044351,1778171243,0x82177ed23d6e7bcedce67ec6a663d19529c48b6915b0...,0x60fa0ce29a7a324b390bf3a4beea0075c81f04cf,0xd9f8bbe437a3423b725c6616c1b543775ecf1110,50000000000000000,198831,202,True,0.05,2026-05-07 16:27:23
1,25044351,1778171243,0x88610c93045ed097e8a707c170745b41347f0d2448ec...,0x152876540325a152e1cf8300875d830b2c8d9043,0xd9f8bbe437a3423b725c6616c1b543775ecf1110,50000000000000000,198831,202,True,0.05,2026-05-07 16:27:23
2,25044351,1778171243,0xa43d5e1c2481ffc2a513bec96722c327e0adb0ed2576...,0x2c23b6c5e3abdf2725f42cb41b5a8b960e480516,0xd9f8bbe437a3423b725c6616c1b543775ecf1110,50000000000000000,198831,202,True,0.05,2026-05-07 16:27:23
3,25044351,1778171243,0xcbc23d9c7307e1775c3a1cf3dcbd1d2513050d7466c9...,0xc44c782ed36e4264292ea29e9abe84a55d71a412,0xd9f8bbe437a3423b725c6616c1b543775ecf1110,50000000000000000,198831,202,True,0.05,2026-05-07 16:27:23
4,25044351,1778171243,0x41a96f2785eca695c5dc50535065bede11627f8ce628...,0x1447cce860310420d347d1cd71d5e6c82271f8e2,0xd9f8bbe437a3423b725c6616c1b543775ecf1110,50000000000000000,198831,202,True,0.05,2026-05-07 16:27:23


In [ ]:
# Pull ERC-20 Transfer logs, chunked (Alchemy caps log queries)
TRANSFER_TOPIC = "0xddf252ad1be2c89b69c2b068fc378daa952ba7f163c4a11628f55a4df523b3ef"
CHUNK = 10  # 100 blocks per getLogs call — safely under any range limit

all_logs = []
t0 = time.time()

for chunk_start in range(START_BLOCK, END_BLOCK + 1, CHUNK):
    chunk_end = min(chunk_start + CHUNK - 1, END_BLOCK)
    logs = rpc("eth_getLogs", [{
        "fromBlock": hex(chunk_start),
        "toBlock":   hex(chunk_end),
        "topics":    [TRANSFER_TOPIC],
    }])
    all_logs.extend(logs)
    print(f"  blocks {chunk_start}-{chunk_end}: {len(logs)} logs (total: {len(all_logs):,})")
    time.sleep(0.05)

print(f"\nDone. {len(all_logs):,} Transfer logs in {time.time()-t0:.0f}s")

  blocks 25044351-25044360: 3922 logs (total: 3,922)
  blocks 25044361-25044370: 3781 logs (total: 7,703)
  blocks 25044371-25044380: 4209 logs (total: 11,912)
  blocks 25044381-25044390: 3560 logs (total: 15,472)
  blocks 25044391-25044400: 3244 logs (total: 18,716)
  blocks 25044401-25044410: 3360 logs (total: 22,076)
  blocks 25044411-25044420: 3417 logs (total: 25,493)
  blocks 25044421-25044430: 3902 logs (total: 29,395)
  blocks 25044431-25044440: 3685 logs (total: 33,080)
  blocks 25044441-25044450: 3838 logs (total: 36,918)
  blocks 25044451-25044460: 3494 logs (total: 40,412)
  blocks 25044461-25044470: 3022 logs (total: 43,434)
  blocks 25044471-25044480: 3840 logs (total: 47,274)
  blocks 25044481-25044490: 3847 logs (total: 51,121)
  blocks 25044491-25044500: 3066 logs (total: 54,187)
  blocks 25044501-25044510: 3419 logs (total: 57,606)
  blocks 25044511-25044520: 3265 logs (total: 60,871)
  blocks 25044521-25044530: 4035 logs (total: 64,906)
  blocks 25044531-25044540: 32

In [ ]:
# Decode logs into a clean DataFrame
def topic_to_addr(topic):
    # topics are 32 bytes; addresses are last 20 bytes (40 hex chars)
    return "0x" + topic[-40:].lower()

log_rows = []
for log in all_logs:
    # ERC-721 Transfers also use the same topic but have 4 topics (the 4th is tokenId).
    # ERC-20 has exactly 3 topics. Filter to ERC-20 only.
    if len(log["topics"]) != 3:
        continue
    log_rows.append({
        "block": int(log["blockNumber"], 16),
        "tx_hash": log["transactionHash"],
        "log_index": int(log["logIndex"], 16),
        "token": log["address"].lower(),
        "from": topic_to_addr(log["topics"][1]),
        "to":   topic_to_addr(log["topics"][2]),
        "value_raw": int(log["data"], 16) if log["data"] != "0x" else 0,
    })

logs_df = pd.DataFrame(log_rows)
# attach timestamp via block → ts mapping from tx_df
block_ts = tx_df.drop_duplicates("block").set_index("block")["ts"]
logs_df["ts"] = logs_df["block"].map(block_ts)
logs_df["dt"] = pd.to_datetime(logs_df["ts"], unit="s")

logs_df.to_csv("../data/processed/token_transfers.csv", index=False)
print(f"Saved {len(logs_df):,} ERC-20 transfers")
print(f"Unique tokens: {logs_df['token'].nunique():,}")
print(f"Unique senders: {logs_df['from'].nunique():,}")
print(f"Unique recipients: {logs_df['to'].nunique():,}")
logs_df.head()